# Text Classification by XGBoost & Others

https://pub.towardsai.net/text-classification-by-xgboost-others-a-case-study-using-bbc-news-articles-5d88e94a9f8

This example is a 5-class classification

## 1.0 - Getting the data

In [3]:
import pandas as pd

bbc_text_df = pd.read_csv('bbc-text.csv')
bbc_text_df.head()

,category,text
0,tech,tv future in the hands of viewers with home th...
1,business,worldcom boss left books alone former worldc...
2,sport,tigers wary of farrell gamble leicester say ...
3,sport,yeading face newcastle in fa cup premiership s...
4,entertainment,ocean s twelve raids box office ocean s twelve...


In [4]:
bbc_text_df.shape

(2225, 2)

## 2.0 Text Cleaning

In this example, we will use use Python ‘gensim’ library for all text cleaning

In [4]:
%pip install gensim

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.6/26.6 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 97.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.2/38.2 MB 15.9 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.2
    Uninstalling scipy-1.16.2:
      Successfully uninstalled scipy-1.16.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tsfre

In [6]:
from gensim import utils
import gensim.parsing.preprocessing as gsp

filters = [
    gsp.strip_tags,
    gsp.strip_punctuation,
    gsp.strip_multiple_whitespaces,
    gsp.strip_numeric,
    gsp.remove_stopwords,
    gsp.strip_short,
    gsp.stem_text
]


def clean_text(s):
    s = s.lower()
    s = utils.to_unicode(s)
    for f in filters:
        s = f(s)
    return s

In [11]:
df_x = bbc_text_df['text']
df_y = bbc_text_df['category']

In [44]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df_y_encoded = le.fit_transform(df_y)

In [45]:
print(le.classes_)

[0 1 2 3 4]


In [14]:
df_x = df_x.apply(clean_text)

In [15]:
from sklearn.model_selection import train_test_split

In [18]:
X_train, X_temp, y_train, y_temp = train_test_split(df_x, df_y_encoded, test_size = 0.2, stratify=df_y_encoded, random_state = 42)

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size = 0.5, stratify=y_temp, random_state=42)

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words="english",
    lowercase=True,
    max_features=50000,        # limit dimensionality (adjust to your RAM)
    ngram_range=(1, 2),        # capture unigrams + bigrams
    min_df=2,                  # ignore rare words appearing in <2 docs
    max_df=0.9,                # drop overly common words appearing in >90% of docs
    sublinear_tf=True,         # use log-scaling on term frequency
    norm="l2",                 # normalize each row vector
)

In [20]:
Xtr = tfidf.fit_transform(X_train)
Xva = tfidf.transform(X_val)
Xte = tfidf.transform(X_test)

In [24]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)
class_to_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
sample_weights = np.array([class_to_weights[c] for c in y_train])

In [26]:
from xgboost import XGBClassifier

clf = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss"
)

In [27]:
clf.fit(Xtr, y_train, sample_weight=sample_weights, eval_set=[(Xva, y_val)], verbose=True)

[0]	validation_0-mlogloss:1.44498
[1]	validation_0-mlogloss:1.31495
[2]	validation_0-mlogloss:1.20881
[3]	validation_0-mlogloss:1.11294
[4]	validation_0-mlogloss:1.03063
[5]	validation_0-mlogloss:0.95718
[6]	validation_0-mlogloss:0.89577
[7]	validation_0-mlogloss:0.83340
[8]	validation_0-mlogloss:0.77649
[9]	validation_0-mlogloss:0.72660
[10]	validation_0-mlogloss:0.68183
[11]	validation_0-mlogloss:0.63939
[12]	validation_0-mlogloss:0.60204
[13]	validation_0-mlogloss:0.56797
[14]	validation_0-mlogloss:0.53759
[15]	validation_0-mlogloss:0.50980
[16]	validation_0-mlogloss:0.48254
[17]	validation_0-mlogloss:0.45701
[18]	validation_0-mlogloss:0.43370
[19]	validation_0-mlogloss:0.41424
[20]	validation_0-mlogloss:0.39510
[21]	validation_0-mlogloss:0.37619
[22]	validation_0-mlogloss:0.36047
[23]	validation_0-mlogloss:0.34658
[24]	validation_0-mlogloss:0.33181
[25]	validation_0-mlogloss:0.32069
[26]	validation_0-mlogloss:0.30823
[27]	validation_0-mlogloss:0.29699
[28]	validation_0-mlogloss:0.2

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [30]:
from sklearn.metrics import accuracy_score, classification_report

# Make predictions on the test set
y_pred = clf.predict(Xte)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

# Print classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=[str(c) for c in le.classes_]))

Accuracy: 0.9507

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.90      0.92        51
           1       0.95      1.00      0.97        39
           2       0.95      0.95      0.95        42
           3       0.98      0.98      0.98        51
           4       0.93      0.93      0.93        40

    accuracy                           0.95       223
   macro avg       0.95      0.95      0.95       223
weighted avg       0.95      0.95      0.95       223



## 4.0 Sample Predictions

In [46]:
# Example of a new text
sample_text = "The stock market experienced a significant downturn today, with major indices falling sharply due to concerns over inflation."

# Clean and transform the sample text using the same functions and vectorizer
cleaned_sample_text = clean_text(sample_text)
transformed_sample_text = tfidf.transform([cleaned_sample_text]) # tfidf expects an iterable

# Make a prediction
predicted_label_encoded = clf.predict(transformed_sample_text)

print(le.classes_)

# Decode the predicted label back to the original category name
predicted_category = le.inverse_transform(predicted_label_encoded)

print(f"Sample Text: {sample_text}")
print(f"Predicted Category: {predicted_category[0]}")

[0 1 2 3 4]
Sample Text: The stock market experienced a significant downturn today, with major indices falling sharply due to concerns over inflation.
Predicted Category: 0
